# Phase 3: Sales Analysis

This notebook calculates the overall KPIs, time-based sales trends, and product performance required for Phase 3.

## 1. Import libraries and load the cleaned dataset

In [ ]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

cleaned_file = project_folder / 'data' / 'processed' / 'cleaned_sales.csv'
sales = pd.read_csv(cleaned_file)

print('Rows loaded:', len(sales))

## 2. Calculate the overall KPIs

In [ ]:
total_revenue = sales['Revenue USD'].sum()
total_profit = sales['Gross Profit USD'].sum()
total_orders = sales['Order Number'].nunique()
total_customers = sales['CustomerKey'].nunique()
average_order_value = total_revenue / total_orders
profit_margin = (total_profit / total_revenue) * 100

kpi_summary = pd.DataFrame({
    'KPI': [
        'Total Revenue',
        'Total Gross Profit',
        'Total Orders',
        'Total Customers',
        'Average Order Value',
        'Gross Margin %',
    ],
    'Value': [
        round(total_revenue, 2),
        round(total_profit, 2),
        total_orders,
        total_customers,
        round(average_order_value, 2),
        round(profit_margin, 2),
    ],
})

display(kpi_summary)

## 3. Analyze yearly sales

In [ ]:
yearly_sales = (
    sales.groupby('Year', as_index=False)['Revenue USD']
    .sum()
    .sort_values('Year')
)

yearly_sales['Growth %'] = yearly_sales['Revenue USD'].pct_change() * 100
yearly_sales['Revenue USD'] = yearly_sales['Revenue USD'].round(2)
yearly_sales['Growth %'] = yearly_sales['Growth %'].round(2)

display(yearly_sales)

In [ ]:
complete_growth_rows = yearly_sales.dropna(subset=['Growth %'])
highest_growth_row = complete_growth_rows.loc[complete_growth_rows['Growth %'].idxmax()]

print('Year with maximum growth:', int(highest_growth_row['Year']))
print('Maximum growth percentage:', highest_growth_row['Growth %'])

## 4. Analyze quarterly sales

In [ ]:
quarterly_sales = (
    sales.groupby(['Year', 'Quarter'], as_index=False)['Revenue USD']
    .sum()
    .sort_values(['Year', 'Quarter'])
)

quarterly_sales['Revenue USD'] = quarterly_sales['Revenue USD'].round(2)
display(quarterly_sales)

## 5. Analyze monthly sales

In [ ]:
monthly_sales = (
    sales.groupby(['Year', 'Month', 'Month Name'], as_index=False)['Revenue USD']
    .sum()
    .sort_values(['Year', 'Month'])
)

monthly_sales['Revenue USD'] = monthly_sales['Revenue USD'].round(2)
display(monthly_sales)

In [ ]:
sales_by_month_name = (
    sales.groupby(['Month', 'Month Name'], as_index=False)['Revenue USD']
    .sum()
    .sort_values('Month')
)

sales_by_month_name['Revenue USD'] = sales_by_month_name['Revenue USD'].round(2)
highest_month = sales_by_month_name.loc[sales_by_month_name['Revenue USD'].idxmax()]
lowest_month = sales_by_month_name.loc[sales_by_month_name['Revenue USD'].idxmin()]

display(sales_by_month_name)
print('Month with highest sales:', highest_month['Month Name'])
print('Month with lowest sales:', lowest_month['Month Name'])

## 6. Check seasonality

Average monthly sales are used so January and February are not overstated by the additional partial year in 2021.

In [ ]:
seasonality = (
    monthly_sales.groupby(['Month', 'Month Name'], as_index=False)['Revenue USD']
    .mean()
    .rename(columns={'Revenue USD': 'Average Monthly Sales'})
    .sort_values('Month')
)

seasonality['Average Monthly Sales'] = seasonality['Average Monthly Sales'].round(2)
highest_average_month = seasonality.loc[seasonality['Average Monthly Sales'].idxmax()]
lowest_average_month = seasonality.loc[seasonality['Average Monthly Sales'].idxmin()]

display(seasonality)
print('Highest average sales month:', highest_average_month['Month Name'])
print('Lowest average sales month:', lowest_average_month['Month Name'])
print('The difference between monthly averages shows a seasonal sales pattern.')

## 7. Analyze sales by category

In [ ]:
category_performance = (
    sales.groupby('Category', as_index=False)
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Gross_Profit=('Gross Profit USD', 'sum'),
    )
    .sort_values('Revenue', ascending=False)
)

category_performance[['Revenue', 'Gross_Profit']] = category_performance[['Revenue', 'Gross_Profit']].round(2)
display(category_performance)

## 8. Analyze sales by subcategory

In [ ]:
subcategory_performance = (
    sales.groupby(['Category', 'Subcategory'], as_index=False)
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Gross_Profit=('Gross Profit USD', 'sum'),
    )
    .sort_values('Revenue', ascending=False)
)

subcategory_performance[['Revenue', 'Gross_Profit']] = subcategory_performance[['Revenue', 'Gross_Profit']].round(2)
display(subcategory_performance)

## 9. Calculate product performance

In [ ]:
product_performance = (
    sales.groupby(['ProductKey', 'Product Name'], as_index=False)
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Gross_Profit=('Gross Profit USD', 'sum'),
    )
)

product_performance[['Revenue', 'Gross_Profit']] = product_performance[['Revenue', 'Gross_Profit']].round(2)

## 10. Find the top 10 products by sales

In [ ]:
top_products_by_sales = product_performance.nlargest(10, 'Revenue')
display(top_products_by_sales)

## 11. Find the top 10 products by gross profit

In [ ]:
top_products_by_profit = product_performance.nlargest(10, 'Gross_Profit')
display(top_products_by_profit)

## 12. Find the bottom 10 products by sales

In [ ]:
bottom_products_by_sales = product_performance.nsmallest(10, 'Revenue')
display(bottom_products_by_sales)

## 13. Find products that have negative gross profit

In [ ]:
loss_making_products = product_performance[product_performance['Gross_Profit'] < 0]

print('Number of loss-making products:', len(loss_making_products))
display(loss_making_products.sort_values('Gross_Profit'))

## 14. Display the Phase 3 answers

In [ ]:
print('PHASE 3 ANSWERS')
print('-' * 50)
print(f'Total company revenue: ${total_revenue:,.2f}')
print(f'Total company gross profit: ${total_profit:,.2f}')
print(f'Average order value: ${average_order_value:,.2f}')
print(f'Gross margin: {profit_margin:.2f}%')
print(f'Month with highest sales: {highest_month["Month Name"]}')
print(f'Highest average sales month: {highest_average_month["Month Name"]}')
print(f'Lowest average sales month: {lowest_average_month["Month Name"]}')
print(f'Year with maximum growth: {int(highest_growth_row["Year"])}')
print(f'Loss-making products: {len(loss_making_products)}')